# Notebook para evaluación de extracción de tablas

En este notebook se prueban extractores de tablas.

In [6]:
from pathlib import Path
import os


BASE_DIR = Path('./')
PDF_DIR = BASE_DIR / 'pdfs_prueba'
GROUND_TRUTH_JSON = PDF_DIR / 'ground_truth' / 'ground_truth_kge.json'
OUTPUT_BASE_DIR = BASE_DIR / 'evaluation_output'
OUTPUT_BASE_DIR.mkdir(parents=True, exist_ok=True)

# Aquí se puede usar: camelot | pdfplumber | tabula | deepdoctection | unstructured | nougat | all
TOOL = 'pdfplumber'

JSON_NAME = 'extraction_result.json'
CSV_NAME = 'extraction_result.csv'
EVALUATION_OUTPUT_NAME = 'evaluation_report_kge.json'
RUN_EVALUATION = True
OVERWRITE = True

POPPLER_BIN = None
if POPPLER_BIN:
    os.environ['PATH'] = f"{POPPLER_BIN}:{os.environ.get('PATH', '')}"

print('TOOL:', TOOL)
print('PDF_DIR:', PDF_DIR)
print('GROUND_TRUTH_JSON:', GROUND_TRUTH_JSON)
print('OUTPUT_BASE_DIR:', OUTPUT_BASE_DIR)
print('JSON_NAME:', JSON_NAME)
print('CSV_NAME:', CSV_NAME)
print('EVALUATION_OUTPUT_NAME:', EVALUATION_OUTPUT_NAME)
print('RUN_EVALUATION:', RUN_EVALUATION)
print('OVERWRITE:', OVERWRITE)
print('PDF_DIR existe:', PDF_DIR.exists())

TOOL: pdfplumber
PDF_DIR: pdfs_prueba
GROUND_TRUTH_JSON: pdfs_prueba/ground_truth/ground_truth_kge.json
OUTPUT_BASE_DIR: evaluation_output
JSON_NAME: extraction_result.json
CSV_NAME: extraction_result.csv
EVALUATION_OUTPUT_NAME: evaluation_report_kge.json
RUN_EVALUATION: True
OVERWRITE: True
PDF_DIR existe: True


## Lo que se necesita

- camelot-py
- pdfplumber
- tabula-py
- deepdoctection
- unstructured
- pandas

Además, se debe tener en cuenta lo siguiente:
- Para `deepdoctection`, se necesita binarios Poppler (`pdftoppm`/`pdftocairo`) en `PATH`.
- Para `tabula`, necesita Java disponible en `PATH`.

In [7]:
import importlib
import shutil

for pkg in ['camelot', 'pdfplumber', 'tabula', 'deepdoctection', 'unstructured', 'pandas', 'pypdf', 'bs4']:
    try:
        importlib.import_module(pkg)
        print(f'{pkg}: OK')
    except Exception as exc:
        print(f'{pkg}: ERROR -> {exc}')

print('pdftoppm:', shutil.which('pdftoppm'))
print('pdftocairo:', shutil.which('pdftocairo'))
print('java:', shutil.which('java'))

camelot: OK
pdfplumber: OK
tabula: OK
deepdoctection: OK
unstructured: OK
pandas: OK
pypdf: OK
bs4: OK
pdftoppm: /usr/bin/pdftoppm
pdftocairo: /usr/bin/pdftocairo
java: /usr/bin/java


In [8]:
import csv
import difflib
import io
import json
import re
import shutil
import subprocess
import tempfile
from datetime import datetime
from pathlib import Path
from time import perf_counter
import pandas as pd
from typing import Any, Dict, List, Optional

SUPPORTED_TOOLS = ['camelot', 'pdfplumber', 'tabula', 'deepdoctection', 'unstructured', 'nougat']

def clean_cell_value(value):
    if value is None:
        return ''
    text = str(value)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def normalize_tool_name(name):
    return clean_cell_value(name).lower().replace(' ', '-')

def df_to_matrix(df):
    return [[clean_cell_value(cell) for cell in row] for row in df.fillna('').astype(str).values.tolist()]

def matrix_to_columns_rows(matrix):
    if not matrix:
        return [], []
    width = max(len(row) for row in matrix) if matrix else 0
    padded = []
    for row in matrix:
        if len(row) < width:
            row = row + [''] * (width - len(row))
        elif len(row) > width:
            row = row[:width]
        padded.append([clean_cell_value(cell) for cell in row])
    if not padded:
        return [], []
    columns = padded[0]
    rows = padded[1:] if len(padded) > 1 else []
    if not any(columns):
        columns = [f'col_{idx + 1}' for idx in range(width)]
        rows = padded
    return columns, rows

def is_plausible_table(columns: List[str], rows: List[List[str]]) -> bool:
    if len(columns) < 2 or len(rows) < 1:
        return False
    non_empty_cells = [clean_cell_value(cell) for row in rows for cell in row if clean_cell_value(cell)]
    if len(non_empty_cells) < max(3, len(columns)):
        return False
    long_cells = sum(1 for cell in non_empty_cells if len(cell) >= 120)
    if non_empty_cells and (long_cells / len(non_empty_cells)) > 0.3:
        return False
    avg_non_empty_per_row = sum(sum(1 for cell in row if clean_cell_value(cell)) for row in rows) / max(1, len(rows))
    if avg_non_empty_per_row < 1.5:
        return False
    avg_words_per_cell = sum(len(cell.split()) for cell in non_empty_cells) / max(1, len(non_empty_cells))
    if len(columns) <= 3 and avg_words_per_cell > 10:
        return False
    return True

def clean_tabula_table(columns, rows):
    if not columns:
        return columns, rows
    width = len(columns)
    padded_rows = []
    for row in rows:
        normalized_row = [clean_cell_value(cell) for cell in row]
        if len(normalized_row) < width:
            normalized_row += [''] * (width - len(normalized_row))
        elif len(normalized_row) > width:
            normalized_row = normalized_row[:width]
        padded_rows.append(normalized_row)
    keep_indices = list(range(width))
    for col_idx in range(width):
        header = clean_cell_value(columns[col_idx])
        header_words = len(header.split())
        col_cells = [row[col_idx] for row in padded_rows if row[col_idx]]
        if not col_cells:
            continue
        narrative_like = sum(1 for cell in col_cells if len(cell) >= 90 or len(cell.split()) >= 16)
        narrative_ratio = narrative_like / max(1, len(col_cells))
        if header_words >= 12 and narrative_ratio >= 0.4:
            keep_indices.remove(col_idx)
    if len(keep_indices) >= 2:
        cleaned_columns = [columns[idx] for idx in keep_indices]
        cleaned_rows = [[row[idx] for idx in keep_indices] for row in padded_rows]
    else:
        cleaned_columns = [clean_cell_value(cell) for cell in columns]
        cleaned_rows = padded_rows
    filtered_rows = []
    for row in cleaned_rows:
        non_empty = [clean_cell_value(cell) for cell in row if clean_cell_value(cell)]
        if not non_empty:
            continue
        row_text = ' '.join(non_empty)
        if re.match(r'^(table|tabla|figure|fig\.?)[\s\.:#-]*\d+', row_text, flags=re.IGNORECASE):
            continue
        token_count = len(row_text.split())
        alpha_only_tokens = sum(1 for token in row_text.split() if token.isalpha())
        alpha_ratio = alpha_only_tokens / max(1, token_count)
        if token_count >= 25 and alpha_ratio > 0.85 and len(non_empty) <= 2:
            continue
        filtered_rows.append(row)
    return cleaned_columns, filtered_rows

def row_matches_header(row, header):
    if not row or not header or len(row) != len(header):
        return False
    row_clean = [clean_cell_value(cell).lower() for cell in row]
    header_clean = [clean_cell_value(cell).lower() for cell in header]
    comparable = [idx for idx, text in enumerate(header_clean) if text]
    if not comparable:
        return False
    exact_matches = sum(1 for idx in comparable if row_clean[idx] == header_clean[idx])
    return exact_matches >= max(1, int(len(comparable) * 0.6))

def maybe_expand_single_row_table(columns, rows):
    if len(rows) != 1:
        return rows
    row = rows[0]
    if len(row) < 2:
        return rows
    tokenized = []
    for cell in row:
        text = clean_cell_value(cell)
        tokens = [token for token in text.split(' ') if token]
        tokenized.append(tokens)
    lengths = [len(tokens) for tokens in tokenized]
    if min(lengths) < 4:
        return rows
    max_len = max(lengths)
    min_len = min(lengths)
    if max_len - min_len > 2 or max_len < 4:
        return rows
    expanded_rows = []
    for i in range(max_len):
        expanded_rows.append([tokens[i] if i < len(tokens) else '' for tokens in tokenized])
    return expanded_rows if len(expanded_rows) > 1 else rows

def split_row_if_delimited(row, expected_cols):
    if expected_cols < 2 or len(row) != 1:
        return row
    text = clean_cell_value(row[0])
    if not text:
        return row
    for delimiter in ['|', '\t', ';']:
        if delimiter not in text:
            continue
        parts = [clean_cell_value(part) for part in text.split(delimiter)]
        non_empty_parts = [part for part in parts if part]
        if len(non_empty_parts) >= max(2, expected_cols - 1):
            if len(parts) < expected_cols:
                parts += [''] * (expected_cols - len(parts))
            elif len(parts) > expected_cols:
                parts = parts[:expected_cols]
            return parts
    return row

def column_header_quality(columns):
    if not columns:
        return 0.0
    cleaned = [clean_cell_value(column) for column in columns]
    non_empty = [column for column in cleaned if column]
    if not non_empty:
        return 0.0
    numeric_only = sum(1 for column in non_empty if re.fullmatch(r'[\d\W]+', column) is not None)
    too_short = sum(1 for column in non_empty if len(column) <= 1)
    score = 1.0
    score -= (len(cleaned) - len(non_empty)) / len(cleaned) * 0.45
    score -= numeric_only / len(non_empty) * 0.35
    score -= too_short / len(non_empty) * 0.20
    return max(0.0, min(1.0, score))

def table_numeric_ratio(table):
    rows = table.get('rows', [])
    if not isinstance(rows, list) or not rows:
        return 0.0
    cells = [str(cell) for row in rows if isinstance(row, list) for cell in row]
    if not cells:
        return 0.0
    numeric_cells = sum(1 for cell in cells if re.search(r'\d', cell))
    return numeric_cells / len(cells)

def has_caption_marker(table):
    title = clean_cell_value(table.get('title', '')).lower()
    return bool(re.search(r'\b(table|tabla)\b', title))

def is_likely_text_artifact(table):
    columns = table.get('columns', [])
    rows = table.get('rows', [])
    if not isinstance(columns, list) or not isinstance(rows, list) or not rows:
        return True
    n_cols = len(columns)
    n_rows = len(rows)
    long_cells = 0
    total_cells = 0
    for row in rows:
        if not isinstance(row, list):
            continue
        for cell in row:
            total_cells += 1
            if len(clean_cell_value(cell)) >= 80:
                long_cells += 1
    long_ratio = (long_cells / total_cells) if total_cells else 0.0
    header_q = column_header_quality([str(column) for column in columns])
    header_values = [clean_cell_value(column) for column in columns]
    empty_headers = sum(1 for column in header_values if not column)
    short_headers = sum(1 for column in header_values if 0 < len(column) <= 2)
    empty_header_ratio = empty_headers / max(1, n_cols)
    short_header_ratio = short_headers / max(1, n_cols)
    bibliography_keywords = ['conference', 'proceedings', 'arxiv', 'journal', 'press', 'doi', 'machine learning']
    bibliography_hits = 0
    row_count = 0
    for row in rows:
        if not isinstance(row, list):
            continue
        row_count += 1
        joined = ' '.join(clean_cell_value(cell).lower() for cell in row)
        if any(keyword in joined for keyword in bibliography_keywords):
            bibliography_hits += 1
    bibliography_ratio = (bibliography_hits / row_count) if row_count else 0.0
    if n_cols >= 10 and header_q < 0.45:
        return True
    if n_cols >= 6 and empty_header_ratio >= 0.35:
        return True
    if n_cols >= 8 and short_header_ratio >= 0.45:
        return True
    if n_rows >= 25 and not has_caption_marker(table):
        return True
    if n_rows >= 15 and not has_caption_marker(table) and bibliography_ratio >= 0.25:
        return True
    if long_ratio > 0.18 and n_cols >= 6:
        return True
    return False

def is_plausible_table_payload(title, columns, rows):
    return is_plausible_table(columns, rows)

def normalize_table(table, page_number, table_index):
    columns = table.get('columns', [])
    rows = table.get('rows', [])
    if not isinstance(columns, list):
        columns = []
    columns = [str(col) if col is not None else '' for col in columns]
    if not isinstance(rows, list):
        rows = []
    normalized_rows = []
    expected_cols = len(columns)
    candidate_lengths = []
    for raw_row in rows:
        if isinstance(raw_row, list):
            row_len = len([cell for cell in raw_row if clean_cell_value(cell)])
            if row_len > 0:
                candidate_lengths.append(row_len)
    if expected_cols == 0 and candidate_lengths:
        expected_cols = max(1, min(20, max(set(candidate_lengths), key=candidate_lengths.count)))
        columns = [f'col_{i + 1}' for i in range(expected_cols)]
    for raw_row in rows:
        if not isinstance(raw_row, list):
            continue
        row = [str(cell) if cell is not None else '' for cell in raw_row]
        if expected_cols == 0:
            expected_cols = len(row)
            columns = [f'col_{i + 1}' for i in range(expected_cols)]
        row = split_row_if_delimited(row, expected_cols)
        if len(row) < expected_cols:
            row.extend([''] * (expected_cols - len(row)))
        elif len(row) > expected_cols:
            row = row[:expected_cols]
        if row_matches_header(row, columns):
            continue
        normalized_rows.append(row)
    if len(columns) <= 3:
        normalized_rows = maybe_expand_single_row_table(columns, normalized_rows)
    if expected_cols == 0:
        columns = []
    return {
        'table_id': f'T{table_index}',
        'title': str(table.get('title', '')).strip(),
        'page': page_number,
        'columns': columns,
        'rows': normalized_rows,
        'notes': str(table.get('notes', '')).strip(),
        'n_rows': len(normalized_rows),
        'n_cols': len(columns),
    }

def html_table_to_matrix(html_text):
    if not html_text:
        return []
    try:
        from bs4 import BeautifulSoup
    except Exception:
        return []
    soup = BeautifulSoup(html_text, 'html.parser')
    rows = soup.find_all('tr')
    if not rows:
        return []
    grid = {}
    max_cols = 0
    max_rows = len(rows)
    for row_idx, row in enumerate(rows):
        cells = row.find_all(['td', 'th'])
        col_idx = 0
        for cell in cells:
            while (row_idx, col_idx) in grid:
                col_idx += 1
            text = clean_cell_value(cell.get_text(' ', strip=True))
            rowspan = int(cell.get('rowspan', 1) or 1)
            colspan = int(cell.get('colspan', 1) or 1)
            for row_span_idx in range(rowspan):
                for col_span_idx in range(colspan):
                    real_row = row_idx + row_span_idx
                    real_col = col_idx + col_span_idx
                    grid[(real_row, real_col)] = text
                    if real_col >= max_cols:
                        max_cols = real_col + 1
            col_idx += colspan
    matrix = []
    for row_idx in range(max_rows):
        current_row = []
        for col_idx in range(max_cols):
            current_row.append(grid.get((row_idx, col_idx), ''))
        if any(clean_cell_value(cell) for cell in current_row):
            matrix.append(current_row)
    return matrix

def save_outputs(output_json_path, output_csv_path, run_payload, cell_rows):
    with Path(output_json_path).open('w', encoding='utf-8') as json_file:
        json.dump(run_payload, json_file, ensure_ascii=False, indent=2)
    csv_headers = ['pdf_name', 'page', 'table_id', 'table_title', 'row_index', 'column_index', 'column_name', 'cell_value']
    with Path(output_csv_path).open('w', encoding='utf-8', newline='') as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=csv_headers)
        writer.writeheader()
        writer.writerows(cell_rows)

def compute_pdf_quality_metrics(tables):
    total_tables = len(tables)
    non_empty_tables = sum(1 for table in tables if int(table.get('n_rows', 0)) > 0)
    empty_tables = total_tables - non_empty_tables
    quality_pct = (non_empty_tables / total_tables * 100.0) if total_tables > 0 else 0.0
    avg_rows = (sum(int(table.get('n_rows', 0)) for table in tables) / total_tables) if total_tables > 0 else 0.0
    return {
        'tables_total': total_tables,
        'tables_with_rows': non_empty_tables,
        'tables_empty': empty_tables,
        'quality_non_empty_rows_pct': round(quality_pct, 2),
        'avg_rows_per_table': round(avg_rows, 2),
    }

def compute_global_quality_metrics(pdf_results):
    total_tables = sum(int(pdf.get('tables_found', 0)) for pdf in pdf_results)
    tables_with_rows = sum(int(pdf.get('quality_metrics', {}).get('tables_with_rows', 0)) for pdf in pdf_results)
    tables_empty = total_tables - tables_with_rows
    quality_pct = (tables_with_rows / total_tables * 100.0) if total_tables > 0 else 0.0
    pdfs_with_any_table = sum(1 for pdf in pdf_results if int(pdf.get('tables_found', 0)) > 0)
    pdfs_with_quality_100 = sum(
        1
        for pdf in pdf_results
        if int(pdf.get('tables_found', 0)) > 0
        and float(pdf.get('quality_metrics', {}).get('quality_non_empty_rows_pct', 0.0)) == 100.0
    )
    return {
        'tables_total': total_tables,
        'tables_with_rows': tables_with_rows,
        'tables_empty': tables_empty,
        'global_quality_non_empty_rows_pct': round(quality_pct, 2),
        'pdfs_with_any_table': pdfs_with_any_table,
        'pdfs_with_quality_100_pct': round((pdfs_with_quality_100 / pdfs_with_any_table * 100.0) if pdfs_with_any_table > 0 else 0.0, 2),
    }

def norm_text(value):
    text = clean_cell_value(value).lower()
    text = text.replace('\u00a0', ' ')
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def similarity(a, b):
    return difflib.SequenceMatcher(None, norm_text(a), norm_text(b)).ratio()

def to_float_if_numeric(value):
    text = norm_text(value)
    if not text or text in {'-', '--', 'na', 'n/a'}:
        return None
    text = text.replace(',', '')
    if re.fullmatch(r'[+-]?\d+(?:\.\d+)?', text) is None:
        return None
    try:
        return float(text)
    except ValueError:
        return None

def canonical_numeric_or_text(value):
    num = to_float_if_numeric(value)
    if num is not None:
        return f'{num:.10f}'.rstrip('0').rstrip('.')
    return norm_text(value)

def table_to_html_like(table):
    cols = table.get('columns', []) if isinstance(table.get('columns'), list) else []
    rows = table.get('rows', []) if isinstance(table.get('rows'), list) else []
    lines = ['<table>', '<thead>', '<tr>']
    for col in cols:
        lines.append(f'<th>{norm_text(col)}</th>')
    lines.extend(['</tr>', '</thead>', '<tbody>'])
    for row in rows:
        lines.append('<tr>')
        if isinstance(row, list):
            for cell in row:
                lines.append(f'<td>{norm_text(cell)}</td>')
        lines.append('</tr>')
    lines.extend(['</tbody>', '</table>'])
    return ''.join(lines)

def cell_set_with_position(table):
    rows = table.get('rows', []) if isinstance(table.get('rows'), list) else []
    cells = set()
    for r_idx, row in enumerate(rows):
        if not isinstance(row, list):
            continue
        for c_idx, cell in enumerate(row):
            val = norm_text(cell)
            if val:
                cells.add((r_idx, c_idx, val))
    return cells

def header_mapping(gt_cols, pred_cols):
    mapping = {}
    used = set()
    for i, gt_col in enumerate(gt_cols):
        best_j = -1
        best_s = -1.0
        for j, pred_col in enumerate(pred_cols):
            if j in used:
                continue
            score = similarity(str(gt_col), str(pred_col))
            if score > best_s:
                best_s = score
                best_j = j
        if best_j >= 0 and best_s >= 0.30:
            mapping[i] = best_j
            used.add(best_j)
    return mapping

def parse_dataset_metric(col_name):
    text = norm_text(col_name)
    if '_' in text:
        left, right = text.split('_', 1)
        return left, right
    return 'unknown', text

def semantic_tuples_from_gt_table(gt_table):
    columns = gt_table.get('evaluation', {}).get('columns', [])
    rows = gt_table.get('rows', [])
    tuples = set()
    if not isinstance(columns, list) or not isinstance(rows, list) or len(columns) < 2:
        return tuples
    for row in rows:
        if not isinstance(row, list) or not row:
            continue
        model = canonical_numeric_or_text(row[0])
        for c_idx in range(1, min(len(columns), len(row))):
            dataset, metric = parse_dataset_metric(str(columns[c_idx]))
            value = canonical_numeric_or_text(row[c_idx])
            if value:
                tuples.add((model, dataset, metric, value))
    return tuples

def semantic_tuples_from_pred_with_gt_schema(gt_table, pred_table):
    gt_cols = gt_table.get('evaluation', {}).get('columns', [])
    pred_cols = pred_table.get('columns', [])
    gt_rows = gt_table.get('rows', [])
    pred_rows = pred_table.get('rows', [])
    tuples = set()
    if not isinstance(gt_cols, list) or not isinstance(pred_cols, list):
        return tuples
    if not isinstance(gt_rows, list) or not isinstance(pred_rows, list):
        return tuples
    col_map = header_mapping(gt_cols, pred_cols)
    for r_idx, gt_row in enumerate(gt_rows):
        if not isinstance(gt_row, list) or r_idx >= len(pred_rows):
            continue
        pred_row = pred_rows[r_idx]
        if not isinstance(pred_row, list):
            continue
        model_val = pred_row[col_map.get(0, 0)] if col_map.get(0, 0) < len(pred_row) else ''
        model = canonical_numeric_or_text(model_val)
        for gt_c_idx in range(1, len(gt_cols)):
            if gt_c_idx not in col_map:
                continue
            pred_c_idx = col_map[gt_c_idx]
            if pred_c_idx >= len(pred_row):
                continue
            dataset, metric = parse_dataset_metric(str(gt_cols[gt_c_idx]))
            value = canonical_numeric_or_text(pred_row[pred_c_idx])
            if value:
                tuples.add((model, dataset, metric, value))
    return tuples

def evaluate_against_ground_truth_documents(gt_payload, pred_payload):
    pred_docs = pred_payload.get('pdf_results', []) if isinstance(pred_payload.get('pdf_results'), list) else []
    gt_docs = gt_payload.get('documents', []) if isinstance(gt_payload.get('documents'), list) else []
    matched_doc_pairs = []
    used_pred = set()
    for gt_doc in gt_docs:
        gt_title = str(gt_doc.get('paper_title', ''))
        best_idx = -1
        best_score = -1.0
        for idx, pred_doc in enumerate(pred_docs):
            if idx in used_pred:
                continue
            pred_title = str(pred_doc.get('pdf_name', ''))
            score = similarity(gt_title, pred_title)
            if score > best_score:
                best_score = score
                best_idx = idx
        if best_idx >= 0:
            used_pred.add(best_idx)
            matched_doc_pairs.append((gt_doc, pred_docs[best_idx], best_score))
    td_tp = 0
    td_fp = 0
    td_fn = 0
    teds_scores = []
    cell_f1_scores = []
    numeric_total = 0
    numeric_correct = 0
    sem_gt_all = set()
    sem_pred_all = set()
    for gt_doc, pred_doc, _ in matched_doc_pairs:
        gt_tables = gt_doc.get('tables', []) if isinstance(gt_doc.get('tables'), list) else []
        pred_tables = pred_doc.get('tables', []) if isinstance(pred_doc.get('tables'), list) else []
        matched_pred_idx = set()
        for gt_table in gt_tables:
            gt_page = gt_table.get('page')
            gt_cols = gt_table.get('evaluation', {}).get('expected_cols', 0)
            gt_rows_n = gt_table.get('evaluation', {}).get('expected_rows', 0)
            best_idx = -1
            best_score = -1.0
            for p_idx, pred_table in enumerate(pred_tables):
                if p_idx in matched_pred_idx:
                    continue
                page_match = 1.0 if pred_table.get('page') == gt_page else 0.0
                pred_cols_n = int(pred_table.get('n_cols', len(pred_table.get('columns', []))))
                pred_rows_n = int(pred_table.get('n_rows', len(pred_table.get('rows', []))))
                cols_score = 1.0 - (abs(pred_cols_n - gt_cols) / max(1, gt_cols, pred_cols_n))
                rows_score = 1.0 - (abs(pred_rows_n - gt_rows_n) / max(1, gt_rows_n, pred_rows_n))
                title_score = similarity(str(gt_table.get('table_id', '')), str(pred_table.get('title', '')))
                match_score = (0.55 * page_match) + (0.25 * cols_score) + (0.15 * rows_score) + (0.05 * title_score)
                if match_score > best_score:
                    best_score = match_score
                    best_idx = p_idx
            if best_idx >= 0 and best_score >= 0.50:
                matched_pred_idx.add(best_idx)
                td_tp += 1
                pred_table = pred_tables[best_idx]
                html_gt = table_to_html_like({'columns': gt_table.get('evaluation', {}).get('columns', []), 'rows': gt_table.get('rows', [])})
                html_pred = table_to_html_like(pred_table)
                teds_scores.append(difflib.SequenceMatcher(None, html_gt, html_pred).ratio())
                gt_cells = cell_set_with_position({'rows': gt_table.get('rows', [])})
                pred_cells = cell_set_with_position(pred_table)
                tp_cells = len(gt_cells & pred_cells)
                p_cells = tp_cells / max(1, len(pred_cells))
                r_cells = tp_cells / max(1, len(gt_cells))
                f1_cells = (2 * p_cells * r_cells / (p_cells + r_cells)) if (p_cells + r_cells) > 0 else 0.0
                cell_f1_scores.append(f1_cells)
                gt_cols_list = gt_table.get('evaluation', {}).get('columns', [])
                gt_rows_list = gt_table.get('rows', [])
                pred_cols_list = pred_table.get('columns', [])
                pred_rows_list = pred_table.get('rows', [])
                col_map = header_mapping(gt_cols_list, pred_cols_list)
                for r_idx, gt_row in enumerate(gt_rows_list):
                    if not isinstance(gt_row, list) or r_idx >= len(pred_rows_list):
                        continue
                    pred_row = pred_rows_list[r_idx]
                    if not isinstance(pred_row, list):
                        continue
                    for gt_c_idx in range(min(len(gt_cols_list), len(gt_row))):
                        gt_num = to_float_if_numeric(gt_row[gt_c_idx])
                        if gt_num is None:
                            continue
                        if gt_c_idx not in col_map:
                            numeric_total += 1
                            continue
                        pred_c_idx = col_map[gt_c_idx]
                        if pred_c_idx >= len(pred_row):
                            numeric_total += 1
                            continue
                        pred_num = to_float_if_numeric(pred_row[pred_c_idx])
                        numeric_total += 1
                        if pred_num is not None and abs(pred_num - gt_num) < 1e-9:
                            numeric_correct += 1
                sem_gt = semantic_tuples_from_gt_table(gt_table)
                sem_pred = semantic_tuples_from_pred_with_gt_schema(gt_table, pred_table)
                sem_gt_all |= sem_gt
                sem_pred_all |= sem_pred
            else:
                td_fn += 1
        td_fp += max(0, len(pred_tables) - len(matched_pred_idx))
    td_precision = td_tp / max(1, td_tp + td_fp)
    td_recall = td_tp / max(1, td_tp + td_fn)
    td_f1 = (2 * td_precision * td_recall / (td_precision + td_recall)) if (td_precision + td_recall) > 0 else 0.0
    sem_tp = len(sem_gt_all & sem_pred_all)
    sem_precision = sem_tp / max(1, len(sem_pred_all))
    sem_recall = sem_tp / max(1, len(sem_gt_all))
    sem_f1 = (2 * sem_precision * sem_recall / (sem_precision + sem_recall)) if (sem_precision + sem_recall) > 0 else 0.0
    timing_summary = pred_payload.get('timing_summary', {}) if isinstance(pred_payload.get('timing_summary'), dict) else {}
    return {
        'summary': {
            'td_f1_proxy': round(td_f1, 4),
            'td_precision_proxy': round(td_precision, 4),
            'td_recall_proxy': round(td_recall, 4),
            'td_note': 'IoU espacial no calculable: ground truth no contiene bounding boxes.',
            'tsr_teds_mean': round(sum(teds_scores) / max(1, len(teds_scores)), 4),
            'tsr_cell_f1_mean': round(sum(cell_f1_scores) / max(1, len(cell_f1_scores)), 4),
            'numeric_cell_accuracy': round((numeric_correct / max(1, numeric_total)), 4),
            'numeric_correct': numeric_correct,
            'numeric_total': numeric_total,
            'semantic_f1': round(sem_f1, 4),
            'semantic_precision': round(sem_precision, 4),
            'semantic_recall': round(sem_recall, 4),
            'semantic_tp': sem_tp,
            'semantic_gt_count': len(sem_gt_all),
            'semantic_pred_count': len(sem_pred_all),
            'processing_total_seconds': round(float(timing_summary.get('total_processing_seconds', 0.0) or 0.0), 4),
        }
    }

def evaluate_against_ground_truth_fallback(gt_payload, pred_payload):
    if isinstance(gt_payload, list):
        gt_entries = gt_payload
    else:
        gt_entries = gt_payload.get('pdf_results', []) if isinstance(gt_payload, dict) else []
    gt_by_pdf = {}
    for entry in gt_entries:
        name = Path(entry.get('pdf_name', '') or entry.get('pdf_path', '')).stem
        gt_by_pdf[name] = entry.get('tables', [])
    pdf_metrics = []
    for pdf_entry in pred_payload.get('pdf_results', []):
        pdf_stem = Path(pdf_entry.get('pdf_name', '') or pdf_entry.get('pdf_path', '')).stem
        pred_tables = pdf_entry.get('tables', [])
        gt_tables = gt_by_pdf.get(pdf_stem, [])
        if not gt_tables:
            pdf_metrics.append({'pdf': pdf_stem, 'note': 'no ground truth', 'n_pred': len(pred_tables)})
            continue
        scores = []
        for gt_t in gt_tables:
            gt_str = json.dumps(gt_t.get('rows', []), ensure_ascii=False)
            best = 0.0
            for pred_t in pred_tables:
                pred_str = json.dumps(pred_t.get('rows', []), ensure_ascii=False)
                best = max(best, difflib.SequenceMatcher(None, gt_str, pred_str).ratio())
            scores.append(best)
        gt_cells = set()
        pred_cells = set()
        for table in gt_tables:
            for row in table.get('rows', []):
                for cell in row:
                    value = clean_cell_value(cell)
                    if value:
                        gt_cells.add(value.lower())
        for table in pred_tables:
            for row in table.get('rows', []):
                for cell in row:
                    value = clean_cell_value(cell)
                    if value:
                        pred_cells.add(value.lower())
        tp = len(gt_cells & pred_cells)
        precision = tp / len(pred_cells) if pred_cells else 0.0
        recall = tp / len(gt_cells) if gt_cells else 0.0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
        pdf_metrics.append({
            'pdf': pdf_stem,
            'n_gt_tables': len(gt_tables),
            'n_pred_tables': len(pred_tables),
            'avg_teds_approx': round(sum(scores) / len(scores), 4) if scores else 0.0,
            'cell_precision': round(precision, 4),
            'cell_recall': round(recall, 4),
            'cell_f1': round(f1, 4),
        })
    teds_vals = [metric['avg_teds_approx'] for metric in pdf_metrics if 'avg_teds_approx' in metric]
    f1_vals = [metric['cell_f1'] for metric in pdf_metrics if 'cell_f1' in metric]
    return {
        'global_avg_teds_approx': round(sum(teds_vals) / len(teds_vals), 4) if teds_vals else None,
        'global_avg_cell_f1': round(sum(f1_vals) / len(f1_vals), 4) if f1_vals else None,
        'per_pdf': pdf_metrics,
    }

def evaluate_against_ground_truth(
    gt_payload: Dict[str, Any],
    pred_payload: Dict[str, Any],
) -> Dict[str, Any]:
    pred_docs = pred_payload.get("pdf_results", []) if isinstance(pred_payload.get("pdf_results"), list) else []
    gt_docs = gt_payload.get("documents", []) if isinstance(gt_payload.get("documents"), list) else []
    matched_doc_pairs: List[tuple] = []
    used_pred = set()
    for gt_doc in gt_docs:
        gt_title = str(gt_doc.get("paper_title", ""))
        best_idx = -1
        best_score = -1.0
        for idx, pred_doc in enumerate(pred_docs):
            if idx in used_pred:
                continue
            pred_title = str(pred_doc.get("pdf_name", ""))
            score = similarity(gt_title, pred_title)
            if score > best_score:
                best_score = score
                best_idx = idx
        if best_idx >= 0:
            used_pred.add(best_idx)
            matched_doc_pairs.append((gt_doc, pred_docs[best_idx], best_score))
    td_tp = 0
    td_fp = 0
    td_fn = 0
    teds_scores: List[float] = []
    cell_f1_scores: List[float] = []
    numeric_total = 0
    numeric_correct = 0
    sem_gt_all = set()
    sem_pred_all = set()
    for gt_doc, pred_doc, _ in matched_doc_pairs:
        gt_tables = gt_doc.get("tables", []) if isinstance(gt_doc.get("tables"), list) else []
        pred_tables = pred_doc.get("tables", []) if isinstance(pred_doc.get("tables"), list) else []
        matched_pred_idx = set()
        for gt_table in gt_tables:
            gt_page = gt_table.get("page")
            gt_cols = gt_table.get("evaluation", {}).get("expected_cols", 0)
            gt_rows_n = gt_table.get("evaluation", {}).get("expected_rows", 0)
            best_idx = -1
            best_score = -1.0
            for p_idx, pred_table in enumerate(pred_tables):
                if p_idx in matched_pred_idx:
                    continue
                page_match = 1.0 if pred_table.get("page") == gt_page else 0.0
                pred_cols_n = int(pred_table.get("n_cols", len(pred_table.get("columns", []))))
                pred_rows_n = int(pred_table.get("n_rows", len(pred_table.get("rows", []))))
                cols_score = 1.0 - (abs(pred_cols_n - gt_cols) / max(1, gt_cols, pred_cols_n))
                rows_score = 1.0 - (abs(pred_rows_n - gt_rows_n) / max(1, gt_rows_n, pred_rows_n))
                title_score = similarity(str(gt_table.get("table_id", "")), str(pred_table.get("title", "")))
                match_score = (0.55 * page_match) + (0.25 * cols_score) + (0.15 * rows_score) + (0.05 * title_score)
                if match_score > best_score:
                    best_score = match_score
                    best_idx = p_idx
            if best_idx >= 0 and best_score >= 0.50:
                matched_pred_idx.add(best_idx)
                td_tp += 1
                pred_table = pred_tables[best_idx]
                html_gt = table_to_html_like({
                    "columns": gt_table.get("evaluation", {}).get("columns", []),
                    "rows": gt_table.get("rows", []),
                })
                html_pred = table_to_html_like(pred_table)
                teds_scores.append(difflib.SequenceMatcher(None, html_gt, html_pred).ratio())
                gt_cells = cell_set_with_position({
                    "rows": gt_table.get("rows", []),
                })
                pred_cells = cell_set_with_position(pred_table)
                tp_cells = len(gt_cells & pred_cells)
                p_cells = tp_cells / max(1, len(pred_cells))
                r_cells = tp_cells / max(1, len(gt_cells))
                f1_cells = (2 * p_cells * r_cells / (p_cells + r_cells)) if (p_cells + r_cells) > 0 else 0.0
                cell_f1_scores.append(f1_cells)
                gt_cols_list = gt_table.get("evaluation", {}).get("columns", [])
                gt_rows_list = gt_table.get("rows", [])
                pred_cols_list = pred_table.get("columns", [])
                pred_rows_list = pred_table.get("rows", [])
                col_map = header_mapping(gt_cols_list, pred_cols_list)
                for r_idx, gt_row in enumerate(gt_rows_list):
                    if not isinstance(gt_row, list) or r_idx >= len(pred_rows_list):
                        continue
                    pred_row = pred_rows_list[r_idx]
                    if not isinstance(pred_row, list):
                        continue
                    for gt_c_idx in range(min(len(gt_cols_list), len(gt_row))):
                        gt_num = to_float_if_numeric(gt_row[gt_c_idx])
                        if gt_num is None:
                            continue
                        if gt_c_idx not in col_map:
                            numeric_total += 1
                            continue
                        pred_c_idx = col_map[gt_c_idx]
                        if pred_c_idx >= len(pred_row):
                            numeric_total += 1
                            continue
                        pred_num = to_float_if_numeric(pred_row[pred_c_idx])
                        numeric_total += 1
                        if pred_num is not None and abs(pred_num - gt_num) < 1e-9:
                            numeric_correct += 1
                sem_gt = semantic_tuples_from_gt_table(gt_table)
                sem_pred = semantic_tuples_from_pred_with_gt_schema(gt_table, pred_table)
                sem_gt_all |= sem_gt
                sem_pred_all |= sem_pred
            else:
                td_fn += 1
        td_fp += max(0, len(pred_tables) - len(matched_pred_idx))
    td_precision = td_tp / max(1, td_tp + td_fp)
    td_recall = td_tp / max(1, td_tp + td_fn)
    td_f1 = (2 * td_precision * td_recall / (td_precision + td_recall)) if (td_precision + td_recall) > 0 else 0.0
    sem_tp = len(sem_gt_all & sem_pred_all)
    sem_precision = sem_tp / max(1, len(sem_pred_all))
    sem_recall = sem_tp / max(1, len(sem_gt_all))
    sem_f1 = (2 * sem_precision * sem_recall / (sem_precision + sem_recall)) if (sem_precision + sem_recall) > 0 else 0.0
    timing_summary = pred_payload.get("timing_summary", {}) if isinstance(pred_payload.get("timing_summary"), dict) else {}
    report = {
        "summary": {
            "td_f1_proxy": round(td_f1, 4),
            "td_precision_proxy": round(td_precision, 4),
            "td_recall_proxy": round(td_recall, 4),
            "td_note": "IoU espacial no calculable: ground truth no contiene bounding boxes.",
            "tsr_teds_mean": round(sum(teds_scores) / max(1, len(teds_scores)), 4),
            "tsr_cell_f1_mean": round(sum(cell_f1_scores) / max(1, len(cell_f1_scores)), 4),
            "numeric_cell_accuracy": round((numeric_correct / max(1, numeric_total)), 4),
            "numeric_correct": numeric_correct,
            "numeric_total": numeric_total,
            "semantic_f1": round(sem_f1, 4),
            "semantic_precision": round(sem_precision, 4),
            "semantic_recall": round(sem_recall, 4),
            "semantic_tp": sem_tp,
            "semantic_gt_count": len(sem_gt_all),
            "semantic_pred_count": len(sem_pred_all),
            "processing_total_seconds": round(float(timing_summary.get("total_processing_seconds", 0.0) or 0.0), 4),
        }
    }
    return report


In [9]:
import argparse
from typing import Any, Dict, List, Optional


# Simple args-like object for notebook context
class SimpleArgs:
    def __init__(self, **kwargs):
        for key, value in kwargs.items():
            setattr(self, key, value)


def extract_with_camelot(pdf_path):
    import camelot

    by_page = {}

    def table_signature(page, columns, rows):
        head_rows = rows[:3]
        signature_payload = {
            'page': page,
            'n_cols': len(columns),
            'n_rows': len(rows),
            'columns': [clean_cell_value(c)[:80] for c in columns],
            'rows': [[clean_cell_value(cell)[:80] for cell in row] for row in head_rows],
        }
        return json.dumps(signature_payload, ensure_ascii=False, sort_keys=True)

    seen_signatures = set()
    for flavor in ('lattice', 'stream'):
        try:
            tables = camelot.read_pdf(str(pdf_path), pages='all', flavor=flavor)
        except Exception:
            continue

        for table in tables:
            page = int(table.page)
            matrix = df_to_matrix(table.df)
            columns, rows = matrix_to_columns_rows(matrix)
            signature = table_signature(page, columns, rows)
            if signature in seen_signatures:
                continue
            seen_signatures.add(signature)
            by_page.setdefault(page, []).append({'title': '', 'columns': columns, 'rows': rows})

    return by_page


def extract_with_pdfplumber(pdf_path):
    import pdfplumber

    by_page = {}
    with pdfplumber.open(str(pdf_path)) as pdf:
        for page_idx, page in enumerate(pdf.pages, start=1):
            tables = page.extract_tables() or []
            for matrix in tables:
                normalized = [[clean_cell_value(cell) for cell in (row or [])] for row in (matrix or [])]
                columns, rows = matrix_to_columns_rows(normalized)
                by_page.setdefault(page_idx, []).append({'title': '', 'columns': columns, 'rows': rows})
    return by_page


def extract_with_tabula(pdf_path):
    import tabula
    from pypdf import PdfReader

    by_page = {}
    n_pages = len(PdfReader(str(pdf_path)).pages)
    for page_num in range(1, n_pages + 1):
        dfs = tabula.read_pdf(str(pdf_path), pages=page_num, multiple_tables=True)
        for df in dfs or []:
            matrix = df_to_matrix(df)
            columns, rows = matrix_to_columns_rows(matrix)
            columns, rows = clean_tabula_table(columns, rows)
            by_page.setdefault(page_num, []).append({'title': '', 'columns': columns, 'rows': rows})
    return by_page


def extract_with_deepdoctection(pdf_path):
    import deepdoctection as dd

    by_page = {}
    analyzer = dd.get_dd_analyzer(config_overwrite=['USE_OCR=False', 'USE_PDF_MINER=True'])
    doc = analyzer.analyze(path=str(pdf_path))
    doc.reset_state()

    for page in doc:
        page_num = int(getattr(page, 'page_number', 0) or 0) + 1
        for table in getattr(page, 'tables', []):
            csv_text = getattr(table, 'csv', '') or ''
            html_text = getattr(table, 'html', '') or ''
            matrix = []

            if csv_text:
                try:
                    df = pd.read_csv(io.StringIO(csv_text))
                    matrix = df_to_matrix(df)
                except Exception:
                    matrix = []

            if not matrix and html_text:
                try:
                    dfs = pd.read_html(io.StringIO(html_text))
                    if dfs:
                        matrix = df_to_matrix(dfs[0])
                except Exception:
                    matrix = []

            columns, rows = matrix_to_columns_rows(matrix)
            by_page.setdefault(page_num, []).append({'title': '', 'columns': columns, 'rows': rows})
    return by_page


def extract_with_unstructured(pdf_path):
    from unstructured.partition.pdf import partition_pdf

    def run_partition_with_fallbacks(pdf_path):
        attempts = [
            {
                'filename': str(pdf_path),
                'strategy': 'hi_res',
                'include_page_breaks': True,
                'infer_table_structure': True,
            },
            {
                'filename': str(pdf_path),
                'strategy': 'hi_res',
                'include_page_breaks': True,
            },
        ]

        last_error = None
        for kwargs in attempts:
            try:
                return partition_pdf(**kwargs)
            except TypeError as exc:
                last_error = exc
                continue
            except Exception as exc:
                last_error = exc
                break
        if last_error is not None:
            raise last_error
        return []

    def text_table_to_matrix(text):
        if not text:
            return []

        raw_lines = [line.strip() for line in str(text).splitlines() if line.strip()]
        candidate_rows = []
        for line in raw_lines:
            if '|' in line:
                parts = [clean_cell_value(part) for part in line.split('|')]
            else:
                parts = [clean_cell_value(part) for part in re.split(r'\s{2,}|	+', line) if clean_cell_value(part)]
            if len(parts) >= 2:
                candidate_rows.append(parts)

        if not candidate_rows and raw_lines:
            tokens = [clean_cell_value(part) for part in re.split(r'\s{2,}|	+|\|', ' '.join(raw_lines)) if clean_cell_value(part)]
            if len(tokens) >= 2:
                width = min(6, max(2, len(tokens) // 2))
                candidate_rows = [tokens[idx: idx + width] for idx in range(0, len(tokens), width)]
                candidate_rows = [row for row in candidate_rows if len(row) >= 2]

        if not candidate_rows:
            return []

        width = max(len(row) for row in candidate_rows)
        matrix = []
        for row in candidate_rows:
            padded = row + [''] * (width - len(row))
            matrix.append(padded[:width])
        return matrix

    def html_to_matrix_with_fallback(html_text, raw_text=''):
        if html_text:
            try:
                dfs = pd.read_html(io.StringIO(html_text))
                if dfs:
                    return [list(dfs[0].columns)] + df_to_matrix(dfs[0])
            except Exception:
                pass

            matrix = html_table_to_matrix(html_text)
            if matrix:
                return matrix

        text_matrix = text_table_to_matrix(raw_text or html_text)
        if text_matrix:
            return text_matrix

        return []

    by_page = {}
    elements = run_partition_with_fallbacks(pdf_path)

    for element in elements:
        if getattr(element, 'category', '') != 'Table':
            continue

        metadata = getattr(element, 'metadata', None)
        page_num = int(getattr(metadata, 'page_number', 1) or 1)
        html_text = (getattr(metadata, 'text_as_html', None) if metadata else None) or ''
        raw_text = getattr(element, 'text', '') or (getattr(metadata, 'text', '') if metadata else '') or ''

        matrix = html_to_matrix_with_fallback(html_text, raw_text)
        columns, rows = matrix_to_columns_rows(matrix)

        if not rows and matrix and len(matrix[0]) >= 2:
            width = len(matrix[0])
            columns = [f'col_{i + 1}' for i in range(width)]
            rows = []
            for raw_row in matrix:
                normalized_row = [clean_cell_value(cell) for cell in raw_row]
                if len(normalized_row) < width:
                    normalized_row += [''] * (width - len(normalized_row))
                rows.append(normalized_row[:width])

        by_page.setdefault(page_num, []).append({
            'title': '',
            'columns': columns,
            'rows': rows,
            'html': html_text,
            'raw_text': raw_text,
        })

    return by_page
def _parse_markdown_tables(markdown_text):
    lines = markdown_text.splitlines()
    blocks = []
    current = []

    for line in lines:
        if '|' in line:
            current.append(line.rstrip())
        else:
            if current:
                blocks.append(current)
                current = []
    if current:
        blocks.append(current)

    tables = []
    separator_re = re.compile(r'^\s*\|?\s*[:\-\s\|]+\|?\s*$')

    for block in blocks:
        if len(block) < 2:
            continue

        def split_row(row):
            row = row.strip()
            if row.startswith('|'):
                row = row[1:]
            if row.endswith('|'):
                row = row[:-1]
            return [clean_cell_value(part) for part in row.split('|')]

        rows = [split_row(row) for row in block]
        if not rows:
            continue

        has_header_sep = len(rows) >= 2 and separator_re.match(block[1]) is not None
        if has_header_sep:
            columns = rows[0]
            data_rows = rows[2:]
        else:
            width = max(len(row) for row in rows)
            columns = [f'col_{idx + 1}' for idx in range(width)]
            data_rows = rows

        width = max(len(columns), max((len(row) for row in data_rows), default=0))
        if width < 2 or not data_rows:
            continue

        if len(columns) < width:
            columns = columns + [f'col_{idx + 1}' for idx in range(len(columns), width)]
        columns = columns[:width]

        padded_rows = []
        for row in data_rows:
            if len(row) < width:
                row = row + [''] * (width - len(row))
            else:
                row = row[:width]
            padded_rows.append(row)

        if not any(any(cell for cell in row) for row in padded_rows):
            continue
        tables.append({'title': '', 'columns': columns, 'rows': padded_rows})

    return tables


def extract_with_nougat(pdf_path):
    nougat_bin = shutil.which('nougat') or shutil.which('nougat-ocr')
    if not nougat_bin:
        raise RuntimeError('No se encontró el ejecutable de nougat (nougat o nougat-ocr) en PATH')

    with tempfile.TemporaryDirectory(prefix='nougat_tables_') as tmp_dir:
        out_dir = Path(tmp_dir)
        commands = [
            [nougat_bin, str(pdf_path), '-o', str(out_dir)],
            [nougat_bin, str(pdf_path), '--out', str(out_dir)],
            [nougat_bin, str(pdf_path), '-o', str(out_dir), '--markdown'],
            [nougat_bin, str(pdf_path), '--out', str(out_dir), '--markdown'],
        ]

        run_result = None
        errors = []
        for cmd in commands:
            proc = subprocess.run(cmd, capture_output=True, text=True, check=False)
            if proc.returncode == 0:
                run_result = proc
                break
            errors.append(f"{' '.join(cmd)} => rc={proc.returncode}: {proc.stderr.strip()[:200]}")

        if run_result is None:
            raise RuntimeError('Fallo al ejecutar nougat. ' + ' | '.join(errors))

        text_candidates = []
        for ext in ('*.mmd', '*.md', '*.markdown', '*.txt'):
            text_candidates.extend(out_dir.rglob(ext))

        markdown_text = ''
        if text_candidates:
            parts = []
            for file_path in sorted(text_candidates):
                try:
                    parts.append(file_path.read_text(encoding='utf-8', errors='ignore'))
                except Exception:
                    continue
            markdown_text = '\n\n'.join(parts)
        elif run_result.stdout:
            markdown_text = run_result.stdout

    if not markdown_text.strip():
        return {}

    tables = _parse_markdown_tables(markdown_text)
    if not tables:
        return {}
    return {1: tables}

def extract_tables_by_tool(tool: str, pdf_path: Path) -> Dict[int, List[Dict[str, Any]]]:
    if tool == "camelot":
        return extract_with_camelot(pdf_path)
    if tool == "pdfplumber":
        return extract_with_pdfplumber(pdf_path)
    if tool == "tabula":
        return extract_with_tabula(pdf_path)
    if tool == "deepdoctection":
        return extract_with_deepdoctection(pdf_path)
    if tool == "unstructured":
        return extract_with_unstructured(pdf_path)
    if tool == "nougat":
        return extract_with_nougat(pdf_path)
    raise ValueError(f"Herramienta no soportada: {tool}")




def run_single_tool(
    args: argparse.Namespace,
    tool: str,
    pdf_files: List[Path],
    output_base_dir: Path,
    ground_truth_payload: Optional[Dict[str, Any]],
) -> Dict[str, Any]:
    run_start = perf_counter()
    tool_dirname = normalize_tool_name(tool)
    output_dir = output_base_dir if output_base_dir.name == tool_dirname else output_base_dir / tool_dirname
    output_dir.mkdir(parents=True, exist_ok=True)

    output_json_path = output_dir / args.json_name
    output_csv_path = output_dir / args.csv_name

    if not args.overwrite and (output_json_path.exists() or output_csv_path.exists()):
        raise FileExistsError(f"Ya existe salida en {output_dir}. Usa --overwrite para sobrescribir.")

    run_payload: Dict[str, Any] = {
        "tool": tool,
        "tool_output_subdir": tool_dirname,
        "generated_at": datetime.utcnow().isoformat() + "Z",
        "input_pdf_dir": str(args.pdf_dir),
        "output_json": str(output_json_path),
        "output_csv": str(output_csv_path),
        "errors": [],
        "filtered_tables": 0,
        "pdf_results": [],
    }

    cell_rows: List[Dict[str, Any]] = []
    total_tables = 0
    total_errors = 0

    for pdf_idx, pdf_path in enumerate(pdf_files, start=1):
        pdf_start = perf_counter()
        print(f"[{tool}] [{pdf_idx}/{len(pdf_files)}] Procesando PDF: {pdf_path.name}")
        pdf_result: Dict[str, Any] = {
            "pdf_name": pdf_path.name,
            "pdf_path": str(pdf_path),
            "tables_found": 0,
            "filtered_tables": 0,
            "errors": [],
            "tables": [],
        }

        table_counter = 0
        by_page: Dict[int, List[Dict[str, Any]]] = {}
        try:
            by_page = extract_tables_by_tool(tool, pdf_path)
        except Exception as exc:
            error_message = str(exc)
            print(f"  - Error en {tool} para {pdf_path.name}: {error_message}")
            pdf_result["errors"].append(error_message)
            run_payload["errors"].append({"pdf_name": pdf_path.name, "error": error_message})
            total_errors += 1

        for page_num in sorted(by_page.keys()):
            for raw_table in by_page.get(page_num, []):
                table_counter += 1
                normalized = normalize_table(
                    {
                        "title": clean_cell_value(raw_table.get("title", "")),
                        "page": page_num,
                        "columns": raw_table.get("columns", []),
                        "rows": raw_table.get("rows", []),
                        "notes": "",
                    },
                    page_number=page_num,
                    table_index=table_counter,
                )
                normalized["pdf_name"] = pdf_path.name
                normalized["source_model"] = tool

                if tool == "unstructured":
                    preview_columns = normalized.get("columns", [])[:4]
                    preview_rows = normalized.get("rows", [])[:2]
                    print(
                        f"[unstructured][debug] page={page_num} table={table_counter} "
                        f"n_cols={normalized.get('n_cols', 0)} n_rows={normalized.get('n_rows', 0)} "
                        f"cols={preview_columns} rows_preview={preview_rows}"
                    )

                if int(normalized.get("n_cols", 0)) < 2 or int(normalized.get("n_rows", 0)) < 1:
                    if tool == "unstructured":
                        print(f"[unstructured][debug] filtered_by=size page={page_num} table={table_counter}")
                    pdf_result["filtered_tables"] += 1
                    run_payload["filtered_tables"] += 1
                    continue

                if tool == "unstructured":
                    unstructured_non_empty = [
                        clean_cell_value(cell)
                        for row in normalized.get("rows", [])
                        if isinstance(row, list)
                        for cell in row
                        if clean_cell_value(cell)
                    ]
                    keep_unstructured = (
                        int(normalized.get("n_cols", 0)) >= 2
                        and len(unstructured_non_empty) >= 2
                    )
                    if not keep_unstructured and not is_plausible_table(normalized.get("columns", []), normalized.get("rows", [])):
                        print(f"[unstructured][debug] filtered_by=plausibility page={page_num} table={table_counter}")
                        pdf_result["filtered_tables"] += 1
                        run_payload["filtered_tables"] += 1
                        continue
                    if keep_unstructured:
                        print(f"[unstructured][debug] kept_by=relaxed_rule page={page_num} table={table_counter}")
                else:
                    if not is_plausible_table(normalized.get("columns", []), normalized.get("rows", [])):
                        pdf_result["filtered_tables"] += 1
                        run_payload["filtered_tables"] += 1
                        continue

                pdf_result["tables"].append(normalized)

                for row_idx, row in enumerate(normalized["rows"], start=1):
                    for col_idx, cell_value in enumerate(row, start=1):
                        column_name = normalized["columns"][col_idx - 1] if col_idx <= len(normalized["columns"]) else ""
                        cell_rows.append(
                            {
                                "pdf_name": pdf_path.name,
                                "page": normalized["page"],
                                "table_id": normalized["table_id"],
                                "table_title": normalized["title"],
                                "row_index": row_idx,
                                "column_index": col_idx,
                                "column_name": column_name,
                                "cell_value": cell_value,
                            }
                        )

        pdf_result["tables_found"] = len(pdf_result["tables"])
        pdf_elapsed_seconds = max(0.0, perf_counter() - pdf_start)
        pdf_result["quality_metrics"] = compute_pdf_quality_metrics(pdf_result["tables"])
        pdf_result["timing"] = {
            "document_processing_seconds": round(pdf_elapsed_seconds, 4),
            "seconds_per_table": round(pdf_elapsed_seconds / max(1, pdf_result["tables_found"]), 4),
        }
        run_payload["pdf_results"].append(pdf_result)
        total_tables += pdf_result["tables_found"]

        save_outputs(output_json_path, output_csv_path, run_payload, cell_rows)

    run_payload["total_pdfs"] = len(pdf_files)
    run_payload["total_tables"] = total_tables
    run_payload["total_errors"] = total_errors
    run_payload["quality_summary"] = compute_global_quality_metrics(run_payload["pdf_results"])
    run_payload["timing_summary"] = {
        "total_processing_seconds": round(max(0.0, perf_counter() - run_start), 4),
    }
    save_outputs(output_json_path, output_csv_path, run_payload, cell_rows)

    evaluation_output_path = output_dir / args.evaluation_output_name
    if args.run_evaluation:
        if ground_truth_payload is not None:
            try:
                evaluation_report = evaluate_against_ground_truth(ground_truth_payload, run_payload)
                with evaluation_output_path.open("w", encoding="utf-8") as eval_file:
                    json.dump(evaluation_report, eval_file, ensure_ascii=False, indent=2)
                print(f"[{tool}] Evaluación guardada en: {evaluation_output_path}")
            except Exception as exc:
                print(f"[{tool}] No se pudo ejecutar evaluación: {exc}")
        else:
            print(f"[{tool}] Ground truth no disponible, evaluación omitida")

    print("=" * 72)
    print("FINALIZADO")
    print(f"Herramienta      : {tool}")
    print(f"PDFs procesados  : {len(pdf_files)}")
    print(f"Tablas totales   : {total_tables}")
    print(f"Errores          : {total_errors}")
    print(f"Filtradas        : {run_payload['filtered_tables']}")
    print(f"JSON             : {output_json_path}")
    print(f"CSV              : {output_csv_path}")
    print("=" * 72)

    if total_errors == len(pdf_files):
        status = "error"
    elif total_errors > 0:
        status = "partial"
    else:
        status = "ok"

    return {
        "tool": tool,
        "status": status,
        "output_dir": str(output_dir),
        "output_json": str(output_json_path),
        "output_csv": str(output_csv_path),
        "evaluation_output": str(evaluation_output_path),
        "total_pdfs": len(pdf_files),
        "total_tables": total_tables,
        "filtered_tables": run_payload["filtered_tables"],
        "total_errors": total_errors,
        "errors": run_payload["errors"],
        "timing_summary": run_payload.get("timing_summary", {}),
    }


# =====================================
# EXECUTE EXTRACTION AND EVALUATION
# =====================================

pdf_dir = Path(PDF_DIR)
output_base_dir = Path(OUTPUT_BASE_DIR)
output_base_dir.mkdir(parents=True, exist_ok=True)

if not pdf_dir.exists() or not pdf_dir.is_dir():
    raise FileNotFoundError(f"No existe el directorio de PDFs: {pdf_dir}")

pdf_files = sorted(pdf_dir.glob("*.pdf"))
if not pdf_files:
    raise FileNotFoundError(f"No se encontraron PDFs en: {pdf_dir}")

gt_payload = None
if RUN_EVALUATION:
    gt_path = Path(GROUND_TRUTH_JSON)
    if gt_path.exists():
        gt_payload = json.loads(gt_path.read_text(encoding='utf-8'))
    else:
        print(f"Ground truth no encontrado: {gt_path}. Evaluación omitida.")

selected_tools = SUPPORTED_TOOLS if TOOL == 'all' else [TOOL]
runs_summary = []

for tool in selected_tools:
    print("\n" + "#" * 72)
    print(f"Iniciando herramienta: {tool}")
    print("#" * 72)
    try:
        args = SimpleArgs(
            json_name=JSON_NAME,
            csv_name=CSV_NAME,
            pdf_dir=str(pdf_dir),
            evaluation_output_name=EVALUATION_OUTPUT_NAME,
            run_evaluation=RUN_EVALUATION,
            overwrite=OVERWRITE,
        )
        
        result = run_single_tool(
            args=args,
            tool=tool,
            pdf_files=pdf_files,
            output_base_dir=output_base_dir,
            ground_truth_payload=gt_payload,
        )
        runs_summary.append(result)
    except Exception as exc:
        print(f"[{tool}] Error fatal: {exc}")
        runs_summary.append({'tool': tool, 'status': 'error', 'error': str(exc)})

if len(selected_tools) == 1:
    aggregate_summary_path = output_base_dir / selected_tools[0] / 'run_summary.json'
else:
    aggregate_summary_path = output_base_dir / 'run_summary.json'

aggregate_payload = {
    'generated_at': datetime.utcnow().isoformat() + 'Z',
    'requested_tool': TOOL,
    'executed_tools': selected_tools,
    'runs': runs_summary,
}
with aggregate_summary_path.open('w', encoding='utf-8') as summary_file:
    json.dump(aggregate_payload, summary_file, ensure_ascii=False, indent=2)

print("\n" + "=" * 72)
print("RESUMEN GLOBAL")
for run in runs_summary:
    status = run.get('status', 'unknown')
    tool = run.get('tool', '?')
    tables = run.get('total_tables', 0)
    errors = run.get('total_errors', 0)
    print(f"- {tool}: {status} (tablas={tables}, errores={errors})")
print(f"Resumen JSON     : {aggregate_summary_path}")
print("=" * 72)




########################################################################
Iniciando herramienta: pdfplumber
########################################################################
[pdfplumber] [1/5] Procesando PDF: Adaptive Margin Ranking Loss for Knowledge Graph Embeddings via a Correntropy Objective Function.pdf
[pdfplumber] [2/5] Procesando PDF: Adversarial Contrastive Estimation.pdf
[pdfplumber] [3/5] Procesando PDF: Binarized Knowledge Graph Embeddings.pdf
[pdfplumber] [4/5] Procesando PDF: HyperKG- Hyperbolic Knowledge Graph Embeddings for Knowledge Base Completion.pdf
[pdfplumber] [5/5] Procesando PDF: Seq2RDF- An end-to-end application for deriving Triples from Natural Language Text.pdf
[pdfplumber] Evaluación guardada en: evaluation_output/pdfplumber/evaluation_report_kge.json
FINALIZADO
Herramienta      : pdfplumber
PDFs procesados  : 5
Tablas totales   : 2
Errores          : 0
Filtradas        : 31
JSON             : evaluation_output/pdfplumber/extraction_result.json
CSV  

In [10]:
import pandas as pd

summary_rows = []
for run in runs_summary:
    summary_rows.append({
        'tool': run.get('tool', ''),
        'status': run.get('status', ''),
        'total_pdfs': run.get('total_pdfs', 0),
        'total_tables': run.get('total_tables', 0),
        'filtered_tables': run.get('filtered_tables', 0),
        'total_errors': run.get('total_errors', 0),
        'output_dir': run.get('output_dir', ''),
    })

pd.DataFrame(summary_rows).sort_values(['tool']).reset_index(drop=True)

,tool,status,total_pdfs,total_tables,filtered_tables,total_errors,output_dir
0,pdfplumber,ok,5,2,31,0,evaluation_output/pdfplumber
